# 02. InSAR Processing: Sentinel-1 TOPS Stack Processing via ISCE2

This notebook documents the end-to-end radar interferometry processing pipeline using the `topsStack` module within ISCE2 (version 2.6.3). The workflow processes 124 Sentinel-1 SLC acquisitions (January 2021 – April 2026) over the Kendal–Semarang–Demak coastal corridor, focused on the subsiding alluvial plain of Sayung District.

### Pipeline Outline:
1. Environment Initialization & `topsStack` System Path Configuration
2. Topographic Data Preparation (SRTM 30 m DEM & Geoid Correction)
3. Multi-Pillar Input Data Validation (SLC Scenes, DEM, and Precise Orbit Ephemerides)
4. Stack Configuration & Execution Plan Generation (`stackSentinel.py`)
5. DEM File-Naming Compatibility & Workspace Harmonization
6. Sequential Execution of InSAR Stack Scripts (`run_01` through `run_16`)

## 1. Environment Initialization & Path Configuration
Mounts Google Drive storage and configures system environment variables (`PATH`, `PYTHONPATH`, `ISCE_HOME`) to interface with the Python 3.11 conda runtime containing ISCE2 and MintPy.

In [ ]:
import os
import sys
import subprocess
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_DIR = '/content/drive/MyDrive/Thesis_Hands_On'
RAW_DATA_DIR = os.path.join(BASE_DIR, 'raw_data')
ORBIT_DIR = os.path.join(RAW_DATA_DIR, 'orbits')
DEM_DIR = os.path.join(BASE_DIR, 'dem')
STACK_DIR = os.path.join(BASE_DIR, 'isce2_stack')
RUN_DIR = os.path.join(STACK_DIR, 'run_files')

for dir_path in [RAW_DATA_DIR, ORBIT_DIR, DEM_DIR, STACK_DIR]:
    os.makedirs(dir_path, exist_ok=True)

site_packages = '/usr/local/lib/python3.11/site-packages'
isce_home = os.path.join(site_packages, 'isce')
isce_share = '/usr/local/share/isce2'
tops_stack_path = os.path.join(isce_share, 'topsStack')

os.environ['ISCE_HOME'] = isce_home
os.environ['PYTHONPATH'] = f"{isce_share}:{site_packages}:{os.environ.get('PYTHONPATH', '')}"
os.environ['PATH'] = f"{isce_home}/bin:{tops_stack_path}:{os.environ.get('PATH', '')}"
sys.path.append(site_packages)
sys.path.append(isce_share)

subprocess.run(['which', 'stackSentinel.py'], check=True)

## 2. Topographic Data Preparation (SRTM 30 m DEM)
Downloads 1-arcsecond global SRTM tiles (`SRTMGL1`) covering the spatial extent of Kendal, Semarang, and Demak via NASA's `earthaccess` client. The raw tiles are stitched and converted from orthometric geoid heights to ellipsoidal WGS84 heights using `dem.py`.

In [ ]:
import earthaccess

auth = earthaccess.login(strategy="interactive")

dem_results = earthaccess.search_data(
    short_name="SRTMGL1",
    bounding_box=(110.10, -7.10, 110.65, -6.80)
)

earthaccess.download(dem_results, DEM_DIR)

os.chdir(DEM_DIR)
!dem.py -a stitch -b -8 -6 110 111 -c -r

dem_wgs84_path = os.path.join(DEM_DIR, 'demLat_S08_S06_Lon_E110_E111.dem.wgs84')
assert os.path.exists(dem_wgs84_path), "Required WGS84 DEM file not found."
print(f"Topographic DEM prepared successfully: {dem_wgs84_path}")

## 3. Multi-Pillar Input Data Validation
Executes rigorous cross-validation checks across all input datasets prior to interferometric stacking:
- **Pillar 1:** Verifies the physical presence and nonzero size of all 124 Sentinel-1 SLC `.zip` archives.
- **Pillar 2:** Validates the presence of the stitched binary DEM and its corresponding ISCE2 XML metadata header.
- **Pillar 3:** Verifies temporal coverage of Precise Orbit Ephemerides (`AUX_POEORB` / `.EOF`), ensuring every satellite acquisition date is bracketed within an active validity window.

In [ ]:
import glob
import datetime

print("Initiating multi-pillar input data validation...")

# Pillar 1
zip_files = sorted(glob.glob(os.path.join(RAW_DATA_DIR, "S1*.zip")))
total_scenes = len(zip_files)
assert total_scenes == 124, f"Dataset mismatch: expected 124 scenes, found {total_scenes}."

corrupted_files = [f for f in zip_files if os.path.getsize(f) == 0]
assert len(corrupted_files) == 0, f"Corrupted 0-byte archives detected: {corrupted_files}"

scene_acquisitions = []
for f in zip_files:
    fname = os.path.basename(f)
    date_str = fname.split('_')[5][:8]
    date_obj = datetime.datetime.strptime(date_str, "%Y%m%d")
    scene_acquisitions.append((date_str, date_obj, fname))

print(f"[Pillar 1 PASSED] All {total_scenes} raw Sentinel-1 SLC archives are verified.")

# Pillar 2:
dem_binary = os.path.join(DEM_DIR, 'demLat_S08_S06_Lon_E110_E111.dem')
dem_metadata = dem_binary + '.xml'

assert os.path.exists(dem_binary), f"DEM binary missing: {dem_binary}"
assert os.path.exists(dem_metadata), f"DEM XML metadata missing: {dem_metadata}"

dem_size_mb = os.path.getsize(dem_binary) / (1024 * 1024)
print(f"[Pillar 2 PASSED] DEM verified: {os.path.basename(dem_binary)} ({dem_size_mb:.2f} MB).")

# Pillar 3:
eof_files = sorted(glob.glob(os.path.join(ORBIT_DIR, "*.EOF")))
assert len(eof_files) > 0, "Orbit directory contains no .EOF files."

orbit_coverage_windows = []
for eof in eof_files:
    parts = os.path.basename(eof).split('_')
    start_utc = datetime.datetime.strptime(parts[6][1:9], "%Y%m%d")
    end_utc = datetime.datetime.strptime(parts[7][:8], "%Y%m%d")
    orbit_coverage_windows.append((start_utc, end_utc))

uncovered_scenes = []
for date_str, date_obj, fname in scene_acquisitions:
    is_covered = any(start.date() <= date_obj.date() <= end.date() for start, end in orbit_coverage_windows)
    if not is_covered:
        uncovered_scenes.append(fname)

assert len(uncovered_scenes) == 0, f"Missing orbit ephemerides for scenes: {uncovered_scenes}"
print(f"[Pillar 3 PASSED] 100%, all the acquisition date have valid precise orbit files.")
print("All data inputs successfully validated. Ready for stack generation.")

## 4. Stack Configuration & Run Script Generation (`stackSentinel.py`)
Generates the complete processing directory structure and bash control scripts (`run_01` to `run_16`).

Key configuration parameters:
- **Reference Date:** `2021-01-06` (primary geometric master)
- **Target Sub-swath & Bounding Box:** Swaths 1, 2, 3 covering latitude `[-7.005, -6.87]` and longitude `[110.39, 110.525]` (cropping isolates burst 1 to 3 in sub-swath IW2)
- **Polarization:** Co-polarized VV channel
- **Interferogram Thresholds:** Maximum temporal baseline of 48 days and spatial baseline below 100 m

In [ ]:
# Execute stackSentinel.py to construct the processing pipeline
cmd_stack = [
    "python",
    os.path.join(tops_stack_path, "stackSentinel.py"),
    "-s", RAW_DATA_DIR,
    "-d", os.path.join(DEM_DIR, "demLat_S08_S06_Lon_E110_E111.dem"),
    "-o", ORBIT_DIR,
    "-a", ORBIT_DIR,
    "-b", "-7.005 -6.87 110.39 110.525",
    "-p", "vv",
    "-W", "interferogram",
    "-c", "2",
    "-w", STACK_DIR
]

result = subprocess.run(
    cmd_stack,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    check=True
)

print("Stack control files generated successfully.")
print("Run files created in workspace:", sorted(os.listdir(RUN_DIR)))

## 5. DEM File-Naming Harmonization in Execution Directory
ISCE2's `topozero` application requires access to the elevation model in the current execution folder (`run_files`) using both underscore and non-underscore naming conventions (`demLat_S08_S06_LonE110_E111.dem` and `_Lon_E110_`). This step clones all binary raster and XML components into `run_files` to prevent runtime `FileNotFound` errors.

In [ ]:
import glob
import shutil

dem_components = glob.glob(os.path.join(DEM_DIR, "demLat_S08_S06_Lon*"))

for src in dem_components:
    fname = os.path.basename(src)
    ext = fname.split('.dem')[-1]

    dest_compact = os.path.join(RUN_DIR, f"demLat_S08_S06_LonE110_E111.dem{ext}")
    dest_underscored = os.path.join(RUN_DIR, f"demLat_S08_S06_Lon_E110_E111.dem{ext}")

    for dest in [dest_compact, dest_underscored]:
        if os.path.lexists(dest):
            os.remove(dest)
        shutil.copy2(src, dest)

print("DEM binaries and headers successfully harmonized in run_files directory.")

## 6. Execution of InSAR Stack Pipeline Scripts
Sequentially executes the generated bash control scripts:
- **`run_01_unpack_topo_reference`**: Unpacks primary reference scene (`20210106`) and generates geometric radar-to-ground look vectors.
- **`run_02_unpack_secondary_slc`**: Unpacks the remaining 123 secondary scenes into sub-swath bursts.
- **`run_03_average_baseline`**: Estimates perpendicular and temporal baselines across all interferometric pairs.
- **`run_04` – `run_08`**: Performs burst overlap extraction, geometric mapping (`geo2rdr`), Enhanced Spectral Diversity (ESD) coregistration, and azimuth misregistration time-series inversion.
- **`run_09` – `run_12`**: Re-samples secondary bursts to the master geometry and merges contiguous burst grids.
- **`run_13` – `run_16`**: Computes multi-looked differential interferograms, filters phase noise using an adaptive Goldstein filter, evaluates spatial coherence, and unwrap phases using SNAPHU with a strict 0.35 coherence threshold.

In [ ]:
os.chdir(RUN_DIR)

pipeline_scripts = [
    "run_01_unpack_topo_reference",
    "run_02_unpack_secondary_slc",
    "run_03_average_baseline",
    "run_04_extract_burst_overlaps",
    "run_05_overlap_geo2rdr",
    "run_06_overlap_resample",
    "run_07_pairs_misreg",
    "run_08_timeseries_misreg",
    "run_09_fullBurst_geo2rdr",
    "run_10_fullBurst_resample",
    "run_11_extract_stack_valid_region",
    "run_12_merge_reference_secondary_slc",
    "run_13_generate_burst_igram",
    "run_14_merge_burst_igram",
    "run_15_filter_coherence",
    "run_16_unwrap"
]

for script in pipeline_scripts:
    script_path = os.path.join(RUN_DIR, script)
    if not os.path.exists(script_path):
        print(f"Skipping {script}: file not found.")
        continue

    print(f"Executing: {script} ...")
    cmd = f"bash {script_path}"
    exec_result = subprocess.run(cmd, shell=True)

    if exec_result.returncode != 0:
        print(f"Pipeline stopped at {script} with exit code {exec_result.returncode}")
        break
    print(f"Completed: {script}")

print("ISCE2 topsStack processing pipeline completed successfully.")